In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_df = pd.read_csv('error_logs.csv')

# Parse timestamps as UTC datetimes (Unix seconds)
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)
error_df['timestamp'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True)

# --- metric.csv summaries ---
metric_overview = pd.DataFrame({
    'total_rows': [metric_df.shape[0]],
    'unique_cmdb_count': [metric_df['cmdb_id'].nunique()],
    'unique_cmdbs_sample_up_to_20': [list(metric_df['cmdb_id'].unique()[:20])]
})

metric_unique_kpis = pd.Series(metric_df['kpi_name'].unique()[:20], name='kpi_name_sample_up_to_20')

metric_group = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(count=('value', 'count'), p95=('value', lambda x: x.quantile(0.95)))
    .sort_values('count', ascending=False)
    .head(20)
)

# --- trace.csv summaries ---
trace_overview = pd.DataFrame({
    'total_rows': [trace_df.shape[0]],
    'unique_cmdb_count': [trace_df['cmdb_id'].nunique()],
    'unique_cmdbs_sample_up_to_20': [list(trace_df['cmdb_id'].unique()[:20])]
})

trace_unique_names = pd.Series(trace_df['trace_name'].unique()[:20], name='trace_name_sample_up_to_20')

trace_group = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(count=('value', 'count'), p95=('value', lambda x: x.quantile(0.95)))
    .sort_values('count', ascending=False)
    .head(20)
)

# --- log.csv summaries ---
log_overview = pd.DataFrame({
    'total_rows': [log_df.shape[0]],
    'unique_cmdb_count': [log_df['cmdb_id'].nunique()],
    'unique_cmdbs_sample_up_to_20': [list(log_df['cmdb_id'].unique()[:20])]
})

log_unique_names = pd.Series(log_df['log_name'].unique()[:20], name='log_name_sample_up_to_20')

log_group = (
    log_df
    .groupby(['cmdb_id', 'log_name'], as_index=False)
    .agg(count=('value', 'count'), p95=('value', lambda x: x.quantile(0.95)))
    .sort_values('count', ascending=False)
    .head(20)
)

# --- error_logs.csv summaries ---
error_overview = pd.DataFrame({
    'total_rows': [error_df.shape[0]],
    'unique_cmdb_count': [error_df['cmdb_id'].nunique()],
    'unique_cmdbs_sample_up_to_20': [list(error_df['cmdb_id'].unique()[:20])]
})

error_sample = error_df[['timestamp', 'cmdb_id', 'message']].head(20)

error_unique_cmdbs = pd.Series(error_df['cmdb_id'].unique()[:20], name='error_cmdb_id_sample_up_to_20')

# Display compact results (as variables; IPython will show them)
metric_overview, metric_unique_kpis, metric_group, trace_overview, trace_unique_names, trace_group, log_overview, log_unique_names, log_group, error_overview, error_sample, error_unique_cmdbs

```
Out[1]:
```
metric_overview, metric_unique_kpis, metric_group, trace_overview, log_overview, log_unique_names, log_group, error_overview
summary = (
    "Summary of telemetry files:\n\n"
    f"- metric.csv: total rows = {int(metric_overview.at[0,'total_rows'])}, "
    f"unique cmdb_id count = {int(metric_overview.at[0,'unique_cmdb_count'])}. "
    f"Sample cmdb_ids (up to 20) = {metric_overview.at[0,'unique_cmdbs_sample_up_to_20'][:20]}. "
    f"Sample kpi_names (up to 20) = {list(metric_unique_kpis.tolist())}.\n"
    f"  Top (cmdb_id, kpi_name) by point count (showing first): {tuple(metric_group[['cmdb_id','kpi_name']].iloc[0])} "
    f"count = {int(metric_group.iloc[0]['count'])}, p95 = {float(metric_group.iloc[0]['p95']):.6g}.\n\n"
    f"- trace.csv: total rows = {int(trace_overview.at[0,'total_rows'])}. (No trace data present.)\n\n"
    f"- log.csv: total rows = {int(log_overview.at[0,'total_rows'])}, "
    f"unique cmdb_id count = {int(log_overview.at[0,'unique_cmdb_count'])}. "
    f"Sample log_names = {list(log_unique_names.tolist())}.\n"
    f"  Example top log group: {tuple(log_group[['cmdb_id','log_name']].iloc[0])} "
    f"count = {int(log_group.iloc[0]['count'])}, p95 = {float(log_group.iloc[0]['p95']):.6g}.\n\n"
    f"- error_logs.csv: total rows = {int(error_overview.at[0,'total_rows'])}. (No error log rows present.)\n\n"
    "Notes: trace and error_logs are empty in this dataset. Metrics include common KPIs such as cpu, mem, "
    "diskio, latency-50/90, socket, workload; many metric groups have 25 points each. Logs show row_count "
    "values (notably high p95 for front-end row_count) while most log.error_count p95 values are 0."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(   total_rows  unique_cmdb_count                       unique_cmdbs_sample_up_to_20
0        1930                 15  [carts, carts-db, catalogue, catalogue-db, fro..., 0           cpu
1        diskio
2    latency-50
3    latency-90
4           mem
5        socket
6      workload
7         error
Name: kpi_name_sample_up_to_20, dtype: object,          cmdb_id    kpi_name  count           p95
0          carts         cpu     25  7.934422e+01
1          carts      diskio     25  5.924902e+04
3          carts  latency-50     25  1.948694e-02
4          carts  latency-90     25  4.712160e-02
5          carts         mem     25  2.219550e+08
6          carts      socket     25  1.305333e+01
10      carts-db         mem     25  8.396829e+07
7          carts    workload     25  8.486297e+00
8       carts-db         cpu     25  4.275059e+00
9       carts-db      diskio     25  2.270569e+06
12     catalogue         cpu     25  1.833097e-01
11      carts-db      socket     25  7.000000e+00
13     catalogue  latency-50     25  3.095734e-03
14     catalogue  latency-90     25  4.772321e-03
46  queue-master         cpu     25  1.407310e+00
15     catalogue         mem     25  6.267372e+06
16     catalogue      socket     25  6.000000e+00
17     catalogue    workload     25  4.260883e+00
18  catalogue-db         cpu     25  1.954904e-01
19  catalogue-db      diskio     25  0.000000e+00,    total_rows  unique_cmdb_count unique_cmdbs_sample_up_to_20
0           0                  0                           [], Series([], Name: trace_name_sample_up_to_20, dtype: object), Empty DataFrame
Columns: [cmdb_id, trace_name, count, p95]
Index: [],    total_rows  unique_cmdb_count                       unique_cmdbs_sample_up_to_20
0         454                 11  [catalogue, front-end, orders, payment, queue-..., 0    log.error_count
1      log.row_count
Name: log_name_sample_up_to_20, dtype: object,          cmdb_id         log_name  count     p95
4      catalogue  log.error_count     25     0.0
5      catalogue    log.row_count     25   167.8
9         orders    log.row_count     25   129.6
8         orders  log.error_count     25     0.0
7      front-end    log.row_count     25  2040.0
6      front-end  log.error_count     25     0.0
20          user  log.error_count     25     0.0
21          user    log.row_count     25   613.4
19      shipping    log.row_count     25    64.8
18      shipping  log.error_count     25     0.0
13       payment    log.row_count     25   104.8
12       payment  log.error_count     25     0.0
15  queue-master    log.row_count     25   585.6
14  queue-master  log.error_count     25     0.0
2       carts-db  log.error_count     24     0.0
3       carts-db    log.row_count     24     2.0
0          carts  log.error_count     23     0.0
1          carts    log.row_count     23   146.8
17    session-db    log.row_count      4     5.0
16    session-db  log.error_count      4     0.0,    total_rows  unique_cmdb_count unique_cmdbs_sample_up_to_20
0           0                  0                           [], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [], Series([], Name: error_cmdb_id_sample_up_to_20, dtype: object))```
```

In [ ]:
# In[2]:

import pandas as pd

# Reuse existing metric_df if present; otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Ensure timestamp parsed
if metric_df['timestamp'].dtype == 'int64' or metric_df['timestamp'].dtype == 'float64':
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Candidate services
candidates = ['carts','catalogue','orders','payment','user']

# 1) Compute global thresholds (count and P95) for candidate components across the entire metric file
thresholds = (
    metric_df[metric_df['cmdb_id'].isin(candidates)]
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        count=('value', 'count'),
        p95=('value', lambda x: x.quantile(0.95))
    )
    .sort_values('count', ascending=False)
    .head(50)
)

# Incident window (UTC)
start = pd.to_datetime('2024-01-19 14:49:49', utc=True)
end = pd.to_datetime('2024-01-19 15:19:49', utc=True)

# 2) Filter metric rows to candidate components within the incident window
window_df = metric_df[
    (metric_df['cmdb_id'].isin(candidates)) &
    (metric_df['timestamp'] >= start) &
    (metric_df['timestamp'] <= end)
].copy()

# Merge with thresholds to attach p95 for each (cmdb_id, kpi_name)
window_with_thresh = window_df.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left', suffixes=('', '_th'))

# Mark anomalies: value >= p95 (treat higher-is-worse only)
# If p95 is NaN (no global threshold because no historic points), treat as non-anomalous
window_with_thresh['anomalous'] = False
mask_has_p95 = window_with_thresh['p95'].notna()
window_with_thresh.loc[mask_has_p95, 'anomalous'] = window_with_thresh.loc[mask_has_p95, 'value'] >= window_with_thresh.loc[mask_has_p95, 'p95']

# Prepare anomaly_rows: anomaly points in window
anomaly_rows = (
    window_with_thresh[window_with_thresh['anomalous']]
    .sort_values('timestamp', ascending=True)
    .assign(timestamp_iso=lambda df: df['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ'))
    .loc[:, ['timestamp_iso', 'cmdb_id', 'kpi_name', 'value', 'p95']]
    .rename(columns={'timestamp_iso': 'timestamp'})
    .head(50)
)

# Prepare anomaly_summary: per (cmdb_id, kpi_name) within window
summary_grp = (
    window_with_thresh
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        anomaly_count=('anomalous', 'sum'),
        total_points_in_window=('value', 'count'),
        earliest_anomaly_ts=('timestamp', lambda s: s[window_with_thresh.loc[s.index, 'anomalous']].min() if s.size>0 else pd.NaT),
        p95=('p95', 'first')
    )
)

# Format earliest_anomaly_ts as ISO string, keep NaT as None
summary_grp['earliest_anomaly_timestamp'] = summary_grp['earliest_anomaly_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
summary_grp.loc[summary_grp['earliest_anomaly_timestamp'].isna(), 'earliest_anomaly_timestamp'] = None
anomaly_summary = (
    summary_grp
    .loc[:, ['cmdb_id', 'kpi_name', 'anomaly_count', 'total_points_in_window', 'earliest_anomaly_timestamp', 'p95']]
    .sort_values('anomaly_count', ascending=False)
    .head(50)
)

# Display compact outputs: thresholds, anomaly_summary, anomaly_rows
thresholds, anomaly_summary, anomaly_rows

```
Out[2]:
```
Summary of findings (plain English):

- Global thresholds computed for candidate services produced 34 (cmdb_id, kpi_name) groups. Most groups have 25 historical points and a computed P95.

- Anomaly timing: many anomalies begin at 2024-01-19T14:52:00Z (the earliest anomaly timestamp for many groups).

- Top anomalous groups (by anomaly count within the incident window):
  - catalogue / socket: 25 anomalies out of 25 points, global P95 = 6.0, earliest anomaly 2024-01-19T14:52:00Z.
  - orders / error: 25/25 anomalies, global P95 = 0.0, earliest 2024-01-19T14:52:00Z (see caution about P95=0 below).
  - user / error: 25/25 anomalies, global P95 = 0.0, earliest 2024-01-19T14:52:00Z.
  - user / socket: 25/25 anomalies, global P95 = 18.0, earliest 2024-01-19T14:52:00Z.
  - payment / socket: 9 anomalies out of 25 points, global P95 = 6.0, earliest 2024-01-19T14:52:00Z.
  - Additional groups (workload, latency, mem, cpu, diskio) show smaller anomaly counts.

- Example anomaly rows (sorted by timestamp): the earliest shown anomaly is
  - 2024-01-19T14:52:00Z, carts / diskio: value = 74061.28, P95 = 59249.02 (value > P95).
  Other examples at 14:52Z include user workload, user latency, catalogue/socket, orders/cpu, payment/socket, etc.

- Important caution: several "error" KPIs have global P95 = 0.0. Because we flagged anomalies as value >= P95, zero-valued P95 means many points with value == 0 appear as "anomalous" by this rule — this likely produces false positives for error-count KPIs. Treat those error-count anomaly results carefully.

- Interpretation / quick takeaway:
  - There is a clear, coincident spike around 14:52Z affecting socket metrics across multiple services (catalogue, user, payment) and a large diskio spike on carts. This pattern suggests network/socket-related problems (socket metrics) plus at least one I/O spike (carts.diskio) around the same time.
  - Because error-count P95 values are zero, error-count "anomalies" are not reliable without a different rule; focus first on socket and diskio spikes as the most actionable signals.

If you want, I can:
- show the full thresholds table,
- expand the anomaly_summary or anomaly_rows beyond the first 50 rows,
- or recompute anomalies treating error-count KPIs differently (e.g., require value > 0).

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id    kpi_name  count           p95
0    carts         cpu     25  7.934422e+01
1    carts      diskio     25  5.924902e+04
3    carts  latency-50     25  1.948694e-02
4    carts  latency-90     25  4.712160e-02
5    carts         mem     25  2.219550e+08
..     ...         ...    ...           ...
29    user  latency-50     25  3.353954e-03
32    user      socket     25  1.800000e+01
31    user         mem     25  8.170491e+06
33    user    workload     25  1.942971e+01
2    carts       error      5  0.000000e+00

[34 rows x 4 columns],       cmdb_id    kpi_name  anomaly_count  total_points_in_window earliest_anomaly_timestamp           p95
12  catalogue      socket             25                      25       2024-01-19T14:52:00Z  6.000000e+00
15     orders       error             25                      25       2024-01-19T14:52:00Z  0.000000e+00
28       user       error             25                      25       2024-01-19T14:52:00Z  0.000000e+00
32       user      socket             25                      25       2024-01-19T14:52:00Z  1.800000e+01
25    payment      socket              9                      25       2024-01-19T14:52:00Z  6.000000e+00
..        ...         ...            ...                     ...                        ...           ...
26    payment    workload              2                      25       2024-01-19T14:52:00Z  2.190640e+00
29       user  latency-50              2                      25       2024-01-19T14:52:00Z  3.353954e-03
30       user  latency-90              2                      25       2024-01-19T14:52:00Z  6.686859e-03
31       user         mem              2                      25       2024-01-19T15:15:00Z  8.170491e+06
33       user    workload              2                      25       2024-01-19T14:52:00Z  1.942971e+01

[34 rows x 6 columns],                 timestamp    cmdb_id    kpi_name         value           p95
1    2024-01-19T14:52:00Z      carts      diskio  74061.275741  59249.020593
32   2024-01-19T14:52:00Z       user    workload     20.957455     19.429707
29   2024-01-19T14:52:00Z       user  latency-90      0.007046      0.006687
28   2024-01-19T14:52:00Z       user  latency-50      0.003396      0.003354
27   2024-01-19T14:52:00Z       user       error      0.000000      0.000000
..                    ...        ...         ...           ...           ...
242  2024-01-19T14:59:00Z  catalogue      socket      6.000000      6.000000
244  2024-01-19T14:59:00Z     orders         cpu      2.660660      1.785310
245  2024-01-19T14:59:00Z     orders       error      0.000000      0.000000
255  2024-01-19T14:59:00Z    payment      socket      6.000000      6.000000
258  2024-01-19T14:59:00Z       user       error      0.000000      0.000000

[50 rows x 5 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd

# Reuse existing log_df if present; otherwise load and parse timestamps
try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv')
    log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)

# Ensure timestamp column is datetime UTC
if not pd.api.types.is_datetime64_any_dtype(log_df['timestamp']):
    log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)
else:
    # ensure tz-aware UTC
    if log_df['timestamp'].dt.tz is None:
        log_df['timestamp'] = log_df['timestamp'].dt.tz_localize('UTC')

# 1) Compute global P95 and P05 per (cmdb_id, log_name) across the entire file
thresholds_log = (
    log_df
    .groupby(['cmdb_id', 'log_name'], as_index=False)['value']
    .agg(p95=lambda x: x.quantile(0.95), p05=lambda x: x.quantile(0.05))
)

# Incident window (UTC)
start = pd.to_datetime('2024-01-19 14:49:49', utc=True)
end = pd.to_datetime('2024-01-19 15:19:49', utc=True)

# 2) Filter log rows to the incident window
window_log = log_df[(log_df['timestamp'] >= start) & (log_df['timestamp'] <= end)].copy()

# Merge thresholds into windowed data
window_log = window_log.merge(thresholds_log, on=['cmdb_id', 'log_name'], how='left')

# Create boolean flags for low/high/error-positive based on global thresholds
window_log['is_low'] = window_log['p05'].notna() & (window_log['value'] <= window_log['p05'])
window_log['is_high'] = window_log['p95'].notna() & (window_log['value'] >= window_log['p95'])
window_log['is_error_positive'] = (window_log['log_name'] == 'log.error_count') & (window_log['value'] > 0)

# Aggregation function per group
def summarize_group(g):
    total = int(g.shape[0])
    count_low = int(g['is_low'].sum())
    earliest_low_ts = g.loc[g['is_low'], 'timestamp'].min()
    earliest_low = None if pd.isna(earliest_low_ts) else earliest_low_ts.strftime('%Y-%m-%dT%H:%M:%SZ')
    count_high = int(g['is_high'].sum())
    earliest_high_ts = g.loc[g['is_high'], 'timestamp'].min()
    earliest_high = None if pd.isna(earliest_high_ts) else earliest_high_ts.strftime('%Y-%m-%dT%H:%M:%SZ')
    # error-positive only relevant for log.error_count
    if g['log_name'].iloc[0] == 'log.error_count':
        count_error_positive = int(g['is_error_positive'].sum())
        earliest_error_ts = g.loc[g['is_error_positive'], 'timestamp'].min()
        earliest_error = None if pd.isna(earliest_error_ts) else earliest_error_ts.strftime('%Y-%m-%dT%H:%M:%SZ')
    else:
        count_error_positive = 0
        earliest_error = None
    p95 = float(g['p95'].iloc[0]) if pd.notna(g['p95'].iloc[0]) else float('nan')
    p05 = float(g['p05'].iloc[0]) if pd.notna(g['p05'].iloc[0]) else float('nan')
    return pd.Series({
        'p95': p95,
        'p05': p05,
        'total_points_in_window': total,
        'count_low': count_low,
        'earliest_low_timestamp': earliest_low,
        'count_high': count_high,
        'earliest_high_timestamp': earliest_high,
        'count_error_positive': count_error_positive,
        'earliest_error_timestamp': earliest_error
    })

# Group and summarize
log_window_summary = (
    window_log
    .groupby(['cmdb_id', 'log_name'], as_index=True)
    .apply(summarize_group)
    .reset_index()
)

# Ranking metric: max of the three counts
log_window_summary['max_relevant_count'] = log_window_summary[['count_low','count_high','count_error_positive']].max(axis=1)

# Order by that metric desc and limit to top 20
log_window_top20 = (
    log_window_summary
    .sort_values('max_relevant_count', ascending=False)
    .loc[:, ['cmdb_id','log_name','p95','p05','total_points_in_window',
             'count_low','earliest_low_timestamp','count_high','earliest_high_timestamp',
             'count_error_positive','earliest_error_timestamp']]
    .head(20)
)

# Display the compact result
log_window_top20

```
Out[3]:
```
Summary of log analysis (plain English):

- What I computed: global P95 and P05 per (cmdb_id, log_name) from the entire log.csv, then examined log rows inside the incident window 2024-01-19 14:49:49 → 2024-01-19 15:19:49 UTC to count low (<= P05) and high (>= P95) occurrences and the earliest such timestamps. I also counted positive error occurrences (value > 0) for log.error_count.

- Key overall observation: many log.error_count groups have global P95 = P05 = 0.0. As a result, within the window every zero-valued point is simultaneously counted as "low" (<= P05) and "high" (>= P95) by the numeric rule. This is an artifact of the thresholding and produces misleading “anomalies” for error counts. For error semantics, value > 0 is a better indicator of real errors.

- Top findings (ordered by relevance from the table):
  - catalogue, shipping, queue-master, payment, orders, front-end, user, carts-db, carts, session-db all show log.error_count with p95=0.0 and p05=0.0. In the incident window these groups have all or most points flagged (count_low = count_high = total_points_in_window). count_error_positive = 0 for all of them (no positive error counts observed).
  - carts-db log.row_count: total 24 points in window, count_low = 21 (earliest low 2024-01-19T14:53:00Z), count_high = 3 (earliest high 2024-01-19T14:57:00Z). This shows a strong low-side shift for carts-db rows.
  - carts log.row_count: 23 points, count_low = 7 (earliest low 2024-01-19T14:55:00Z), count_high = 2 (earliest high 2024-01-19T15:08:00Z).
  - catalogue, front-end, queue-master, payment, orders, shipping, user row_count each show small numbers of low and high points (typically count_low = 2 and count_high = 2), with earliest low timestamps at ~2024-01-19T14:52:00Z and earliest highs between ~15:04–15:11Z.
  - session-db has only 4 points in-window and both low and high counts = 4 (artifact of equal P05/P95 for that small series).

- Timing pattern: many earliest low/high events originate around 2024-01-19T14:52:00Z (several services), with some later high events appearing ~15:04–15:11Z. carts-db lows start around 14:53Z and carts row_count lows around 14:55Z.

- Practical interpretation and recommendation:
  - Treat log.error_count results with caution: the threshold rule (>=P95 or <=P05) is not meaningful when P95/P05 = 0. Instead, consider error as significant when log.error_count > 0. In this data no positive error_count was observed in-window.
  - The row_count signals show mixture of low and later high counts across multiple services. carts-db shows a pronounced low-side drop (21 of 24 points low), which could indicate reduced throughput or backpressure affecting that component.
  - Correlate these log findings with metric anomalies already found (socket spikes and carts.diskio spike at ~14:52Z) to prioritize investigation: socket/disk I/O + reduced row_count on carts-db point toward network/socket or I/O impact affecting service throughput.

If you want, I can:
- re-run the log anomaly counts treating log.error_count as "anomalous only if value > 0",
- expand the top table or show the raw windowed log points for any specific service.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id         log_name     p95     p05  total_points_in_window  count_low earliest_low_timestamp  count_high earliest_high_timestamp  count_error_positive earliest_error_timestamp
4      catalogue  log.error_count     0.0     0.0                      25         25   2024-01-19T14:52:00Z          25    2024-01-19T14:52:00Z                     0                     None
18      shipping  log.error_count     0.0     0.0                      25         25   2024-01-19T14:52:00Z          25    2024-01-19T14:52:00Z                     0                     None
14  queue-master  log.error_count     0.0     0.0                      25         25   2024-01-19T14:52:00Z          25    2024-01-19T14:52:00Z                     0                     None
12       payment  log.error_count     0.0     0.0                      25         25   2024-01-19T14:52:00Z          25    2024-01-19T14:52:00Z                     0                     None
8         orders  log.error_count     0.0     0.0                      25         25   2024-01-19T14:52:00Z          25    2024-01-19T14:52:00Z                     0                     None
6      front-end  log.error_count     0.0     0.0                      25         25   2024-01-19T14:52:00Z          25    2024-01-19T14:52:00Z                     0                     None
20          user  log.error_count     0.0     0.0                      25         25   2024-01-19T14:52:00Z          25    2024-01-19T14:52:00Z                     0                     None
2       carts-db  log.error_count     0.0     0.0                      24         24   2024-01-19T14:53:00Z          24    2024-01-19T14:53:00Z                     0                     None
0          carts  log.error_count     0.0     0.0                      23         23   2024-01-19T14:53:00Z          23    2024-01-19T14:53:00Z                     0                     None
3       carts-db    log.row_count     2.0     1.0                      24         21   2024-01-19T14:53:00Z           3    2024-01-19T14:57:00Z                     0                     None
1          carts    log.row_count   146.8    25.0                      23          7   2024-01-19T14:55:00Z           2    2024-01-19T15:08:00Z                     0                     None
16    session-db  log.error_count     0.0     0.0                       4          4   2024-01-19T14:57:00Z           4    2024-01-19T14:57:00Z                     0                     None
17    session-db    log.row_count     5.0     5.0                       4          4   2024-01-19T14:57:00Z           4    2024-01-19T14:57:00Z                     0                     None
5      catalogue    log.row_count   167.8   132.2                      25          2   2024-01-19T14:52:00Z           2    2024-01-19T15:04:00Z                     0                     None
7      front-end    log.row_count  2040.0  1566.0                      25          2   2024-01-19T14:52:00Z           2    2024-01-19T15:04:00Z                     0                     None
15  queue-master    log.row_count   585.6   446.4                      25          2   2024-01-19T14:52:00Z           2    2024-01-19T15:11:00Z                     0                     None
13       payment    log.row_count   104.8    84.8                      25          2   2024-01-19T14:52:00Z           2    2024-01-19T15:11:00Z                     0                     None
9         orders    log.row_count   129.6    99.2                      25          2   2024-01-19T14:52:00Z           2    2024-01-19T15:11:00Z                     0                     None
19      shipping    log.row_count    64.8    49.6                      25          2   2024-01-19T14:52:00Z           2    2024-01-19T15:11:00Z                     0                     None
21          user    log.row_count   613.4   477.2                      25          2   2024-01-19T14:52:00Z           2    2024-01-19T15:06:00Z                     0                     None```
```